### Cell 1 - Imports and Seeds

In [ ]:
import random
import numpy as np
import torch

from dataset_utils.constants import Cresci17SetTypes
from dataset_utils.Cresci17 import Cresci17
from dataset_utils.InterleavedIterableDataset import (
    InterleavedIterableDataset,
)

from feature_pipeline import FeaturePipeline
from experiment_runner import (
    TaskDefinition,
    run_continual_experiment,
)
from classifier_strategies import MulticlassPNNStrategy


RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

### Cell 2 — Cresci multiclass tasks

In [ ]:
DATASET_ROOT = "./datasets"


def identity_label(label: str) -> str:
    return str(label)


def make_initial_cresci_dataset(mode: str):
    """
    Task 0:
    genuine_user vs fake_followers
    """

    return InterleavedIterableDataset(
        datasets=[
            Cresci17(
                subset_type=Cresci17SetTypes.GENUINE_USER,
                mode=mode,
                root=DATASET_ROOT,
            ),
            Cresci17(
                subset_type=Cresci17SetTypes.FAKE_FOLLOWER,
                mode=mode,
                root=DATASET_ROOT,
            ),
        ],
        mode="RoundRobin",
    )


def make_cresci_task(
    subset_type: Cresci17SetTypes,
) -> TaskDefinition:
    """
    Later tasks contain one newly introduced bot class.
    """

    return TaskDefinition(
        name=f"Cresci17_{subset_type.name}",

        train_factory=lambda subset_type=subset_type: Cresci17(
            subset_type=subset_type,
            mode="train",
            root=DATASET_ROOT,
        ),

        test_factory=lambda subset_type=subset_type: Cresci17(
            subset_type=subset_type,
            mode="test",
            root=DATASET_ROOT,
        ),

        label_transform=identity_label,
    )


TASK_DEFINITIONS = [
    TaskDefinition(
        name="Cresci17_Initial_Genuine_vs_FakeFollower",

        train_factory=lambda: make_initial_cresci_dataset(
            "train"
        ),

        test_factory=lambda: make_initial_cresci_dataset(
            "test"
        ),

        label_transform=identity_label,
    ),

    make_cresci_task(
        Cresci17SetTypes.SOCIAL_SPAM_1
    ),

    make_cresci_task(
        Cresci17SetTypes.TRADITIONAL_SPAM_1
    ),
]


for task_index, task in enumerate(TASK_DEFINITIONS):
    print(f"Task {task_index}: {task.name}")

### Configure and run experiment

In [ ]:
FEATURE_PIPELINE_CONFIG = {
    "embedding_model": "distilbert-base-uncased",

    "max_tweets_per_user": 20,
    "tweet_batch_size": 32,
    "max_token_length": 128,

    "umap_components": 15,
    "umap_neighbors": 20,
    "umap_min_dist": 0.1,
    "umap_metric": "cosine",

    "random_seed": RANDOM_SEED,
}


STRATEGY_CONFIG = {
    "epochs": 15,
    "learning_rate": 1e-2,
    "dropout_p": 0.1,

    "replay_per_class": 200,
    "balanced_samples_per_class": 200,

    "hdbscan_min_cluster_size": 10,
    "hdbscan_current_fraction": 0.80,

    # True = expand when labels show that a new class arrived.
    # HDBSCAN remains logged as a diagnostic.
    "use_intervention_override": True,

    "eval_batch_size": 512,
    "random_seed": RANDOM_SEED,
}


feature_pipeline = FeaturePipeline(
    config=FEATURE_PIPELINE_CONFIG,
    device=device,
)

strategy = MulticlassPNNStrategy()

results = run_continual_experiment(
    task_definitions=TASK_DEFINITIONS,
    feature_pipeline=feature_pipeline,
    strategy=strategy,
    strategy_config=STRATEGY_CONFIG,
    device=device,
)

strategy_state = results["strategy_state"]
experiment_manager = results["experiment_manager"]
all_step_metrics = results["all_step_metrics"]